# 04 · Encapsulation and abstraction

**Day 1 · about 60 minutes live**

In the warm-up notebook (00), scoring next month's customers produced different results depending on who else was in the file. Time to solve that mystery. The fix is an object that **learns** something from the training data, **keeps** it safe, and **replays** it on every later batch. Building it correctly needs two ideas:

- **Encapsulation**: an object controls its own state. Callers use a small public interface and do not reach into the internals.
- **Abstraction**: a base class states *what* every preprocessing step must be able to do (`fit`, `transform`) without saying *how*.

**Learning goals.** By the end you can:

- explain why a preprocessing step must learn its parameters in `fit` and reuse them in `transform` [Understand, level 2]
- separate public interface from internal state with naming conventions, read-only properties and validating setters [Apply, level 2]
- define an abstract base class and implement a subclass that fulfils its contract [Apply, level 2]
- recognise data that leaks out of an object through shared references, and prevent it [Analyse, level 3]

**Before you start:** notebooks 01 to 03.

Exercises come in three levels: **Core** (everyone), **Stretch** (if you finish early) and **Challenge** (for experienced programmers). Check cells fail until you solve the exercise above them. That is expected.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = Path("../data/raw/telco_churn.csv")

customers = pd.read_csv(DATA_PATH).set_index("customerID")
churned = (customers.pop("Churn") == "Yes").astype(int)

CATEGORICAL = ["Contract", "InternetService", "PaymentMethod"]
NUMERIC = ["tenure", "MonthlyCharges"]

train_customers, new_customers, y_train, y_new = train_test_split(
    customers, churned, test_size=0.2, random_state=42, stratify=churned
)
print(
    f"{len(train_customers)} customers to learn from, {len(new_customers)} 'new' customers to score"
)

## 1. The mystery from notebook 00, solved

Here is the bug in miniature. Four training customers, and next month a batch of two new customers who both happen to have a two-year contract.

In [ ]:
from sklearn.preprocessing import LabelEncoder

train = pd.DataFrame({"Contract": ["Month-to-month", "One year", "Two year", "Month-to-month"]})
next_month = pd.DataFrame({"Contract": ["Two year", "Two year"]})

encoder = LabelEncoder()
print(
    "codes at training time:",
    dict(zip(train["Contract"], encoder.fit_transform(train["Contract"]).tolist(), strict=True)),
)
print(
    "codes next month:      ",
    dict(
        zip(
            next_month["Contract"],
            encoder.fit_transform(next_month["Contract"]).tolist(),
            strict=True,
        )
    ),
)

print()
print("get_dummies at training time:", pd.get_dummies(train["Contract"]).columns.tolist())
print("get_dummies next month:      ", pd.get_dummies(next_month["Contract"]).columns.tolist())

"Two year" was code 2 when the model was trained, and becomes code 0 next month, because the encoder was fitted again on the new batch and only saw one category. The model receives a number it interprets as "Month-to-month". No error, no warning, wrong predictions. `get_dummies` has the same root cause; there the column set changes, which is at least easier to notice.

The cause is not a typo. The encoding is **learned from data**, and it was learned from the wrong data. The fix is a pattern you know from scikit-learn:

- `fit(train)` learns the parameters (here: which categories exist) and **stores them in the object**,
- `transform(any_batch)` applies the stored parameters and learns nothing.

That stored knowledge is the object's **state**. The rest of this notebook is about protecting it.

## 2. Encapsulation: who is allowed to touch the state?

A first one-hot encoder with the fit/transform split. Its learned categories are a plain public attribute.

In [ ]:
class OneHotEncoder:
    """One column per category seen during fit (first version)."""

    def __init__(self, columns: list[str]):
        self.columns = list(columns)
        self.categories = {}  # learned in fit: column -> sorted list of categories
        self.fitted = False

    def fit(self, X: pd.DataFrame) -> "OneHotEncoder":
        self.categories = {col: sorted(X[col].unique()) for col in self.columns}
        self.fitted = True
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if not self.fitted:
            raise RuntimeError("OneHotEncoder must be fitted before transform()")
        out = X.drop(columns=self.columns)
        for col, cats in self.categories.items():
            for cat in cats:
                out[f"{col}={cat}"] = (X[col] == cat).astype(int)
        return out

    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return self.fit(X).transform(X)


encoder = OneHotEncoder(["Contract"]).fit(train)
encoder.transform(next_month)

The two-year customers now get exactly the columns the model was trained with. Mystery solved.

Now a colleague builds a dashboard filter. They only want to offer the short contracts, so they take the list from the encoder and remove one entry. A harmless line, in a different file, written weeks later:

In [ ]:
contract_options = encoder.categories["Contract"]
contract_options.remove("Two year")  # "we only need the short contracts in the dropdown"

encoder.transform(next_month)

The column `Contract=Two year` has vanished from the model input. `contract_options` was not a copy: it was the **same list object** the encoder uses internally. Python passes references, so whoever holds a reference can change the encoder's state.

Encapsulation addresses this in two steps:

1. **Mark internals as internal.** A leading underscore (`_categories`) is Python's convention for "implementation detail, do not use from outside". It is a convention, not a lock: Python still lets you access `encoder._categories`. But it tells every reader and every IDE that this attribute is not part of the interface and may change without notice.
2. **Expose what callers legitimately need through a controlled interface.** A **property** looks like an attribute to the caller but runs a method. Here it returns a *copy*, so nobody can corrupt the original by accident.

In [ ]:
class OneHotEncoder:
    """One column per category seen during fit."""

    def __init__(self, columns: list[str]):
        self.columns = list(columns)
        self._categories: dict[str, list[str]] = {}
        self._fitted = False

    @property
    def categories(self) -> dict[str, list[str]]:
        """Learned categories per column (a copy: changing it cannot affect the encoder)."""
        return {col: list(cats) for col, cats in self._categories.items()}

    @property
    def output_columns(self) -> list[str]:
        return [f"{col}={cat}" for col, cats in self._categories.items() for cat in cats]

    @property
    def is_fitted(self) -> bool:
        return self._fitted

    def fit(self, X: pd.DataFrame) -> "OneHotEncoder":
        self._categories = {col: sorted(X[col].unique()) for col in self.columns}
        self._fitted = True
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if not self._fitted:
            raise RuntimeError("OneHotEncoder must be fitted before transform()")
        out = X.drop(columns=self.columns)
        for col, cats in self._categories.items():
            for cat in cats:
                out[f"{col}={cat}"] = (X[col] == cat).astype(int)
        return out

    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return self.fit(X).transform(X)


encoder = OneHotEncoder(["Contract"]).fit(train)
contract_options = encoder.categories["Contract"]
contract_options.remove("Two year")  # the same dashboard code as before

print("dashboard list:", contract_options)
print("encoder still knows:", encoder.categories["Contract"])
print("model input columns:", encoder.transform(next_month).columns.tolist())

Read-only properties also refuse assignment, which catches the "I'll just set it quickly" mistake:

In [ ]:
try:
    encoder.is_fitted = False
except AttributeError as exc:
    print("AttributeError:", exc)

### What about double underscores?

`self.__secret` (two leading underscores) triggers **name mangling**: Python renames the attribute to `_ClassName__secret`. Its purpose is to avoid name clashes in subclasses, not to hide data. It is rarely needed in data science code; a single underscore is the norm.

In [ ]:
class Account:
    def __init__(self):
        self.__balance = 100


acc = Account()
print(vars(acc))  # the attribute was renamed
print(acc._Account__balance)  # and is still reachable

### Check your understanding

Which of these lines does Python prevent?

```python
encoder._categories = {}                 # (a)
encoder.categories = {}                  # (b)
encoder.categories["Contract"].clear()   # (c)
```

<details><summary>Answer</summary>

Only (b) raises: `categories` is a property without a setter, so assignment fails with `AttributeError`. Line (a) works: the underscore is a convention, and the person who writes it has been warned. Line (c) runs without error, but it only clears the *copy* that the property returned, so the encoder is unaffected. Encapsulation in Python is about making the right thing easy and the wrong thing obviously wrong, not about locks.
</details>

## 3. Properties with validation

Properties can also have a **setter**: a method that runs when someone assigns to the attribute. That is the right place to protect an invariant.

The retention team decides how many customers to call by choosing a probability threshold. Changing it after training is a legitimate business decision, so it should be settable. But it must stay a probability. A threshold of `30` (someone typed a percentage) would silently flag nobody.

First we train a model on the real data, using our encoder for the categorical columns.

In [ ]:
from sklearn.linear_model import LogisticRegression

churn_encoder = OneHotEncoder(CATEGORICAL).fit(train_customers)


def model_input(batch: pd.DataFrame) -> pd.DataFrame:
    """Columns the model sees: numeric features plus the learned one-hot columns."""
    return churn_encoder.transform(batch)[NUMERIC + churn_encoder.output_columns]


model = LogisticRegression(max_iter=1000).fit(model_input(train_customers), y_train)
print(f"{len(churn_encoder.output_columns)} one-hot columns, {len(NUMERIC)} numeric columns")

In [ ]:
class ChurnFlagger:
    """Flag customers whose churn probability reaches a threshold."""

    def __init__(self, model, threshold: float = 0.5):
        self._model = model
        self.threshold = threshold  # goes through the setter below, so it is validated too

    @property
    def threshold(self) -> float:
        return self._threshold

    @threshold.setter
    def threshold(self, value: float) -> None:
        if not 0 < value < 1:
            raise ValueError(f"threshold must be a probability between 0 and 1, got {value}")
        self._threshold = value

    def flag(self, X: pd.DataFrame) -> np.ndarray:
        return (self._model.predict_proba(X)[:, 1] >= self._threshold).astype(int)


flagger = ChurnFlagger(model)
X_new = model_input(new_customers)
print("threshold 0.5 flags", flagger.flag(X_new).sum(), "customers")

flagger.threshold = 0.3  # the team has budget for more calls
print("threshold 0.3 flags", flagger.flag(X_new).sum(), "customers")

try:
    flagger.threshold = 30  # meant 30 %
except ValueError as exc:
    print("ValueError:", exc)

Notice that `__init__` assigns `self.threshold`, not `self._threshold`. That way the constructor uses the same validation as every later assignment, and there is exactly one place where the rule lives.

## 4. Abstraction: a contract for every preprocessing step

A real pipeline has several steps: encode categories, fill gaps, scale numbers. They all should offer the same interface (`fit`, `transform`, `fit_transform`), so that the code using them does not care which step it has (that is polymorphism, notebook 05).

An **abstract base class** (ABC) writes this contract down:

- `@abstractmethod` marks methods every subclass *must* implement.
- Ordinary methods in the base class are shared by all subclasses. `fit_transform` is the same for every step, so it is written once.
- Python refuses to create an object of a class that still has abstract methods. A forgotten method fails at creation time, not halfway through a nightly job.

In [ ]:
from abc import ABC, abstractmethod


class Transformer(ABC):
    """Contract for a preprocessing step: learn in fit, replay in transform."""

    @abstractmethod
    def fit(self, X: pd.DataFrame) -> "Transformer":
        """Learn parameters from training data and return self."""

    @abstractmethod
    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """Apply the learned parameters. Must not learn anything, must not modify X."""

    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return self.fit(X).transform(X)


try:
    Transformer()
except TypeError as exc:
    print("TypeError:", exc)


class HalfDoneScaler(Transformer):
    def fit(self, X):
        return self

    # transform() forgotten


try:
    HalfDoneScaler()
except TypeError as exc:
    print("TypeError:", exc)

Our encoder becomes a subclass. Nothing in its body changes except that `fit_transform` is now inherited from `Transformer`.

In [ ]:
class OneHotEncoder(Transformer):
    """One column per category seen during fit. Same output columns for every batch."""

    def __init__(self, columns: list[str]):
        self.columns = list(columns)
        self._categories: dict[str, list[str]] = {}
        self._fitted = False

    @property
    def categories(self) -> dict[str, list[str]]:
        return {col: list(cats) for col, cats in self._categories.items()}

    @property
    def output_columns(self) -> list[str]:
        return [f"{col}={cat}" for col, cats in self._categories.items() for cat in cats]

    @property
    def is_fitted(self) -> bool:
        return self._fitted

    def fit(self, X: pd.DataFrame) -> "OneHotEncoder":
        self._categories = {col: sorted(X[col].unique()) for col in self.columns}
        self._fitted = True
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if not self._fitted:
            raise RuntimeError("OneHotEncoder must be fitted before transform()")
        out = X.drop(columns=self.columns)
        for col, cats in self._categories.items():
            for cat in cats:
                out[f"{col}={cat}"] = (X[col] == cat).astype(int)
        return out


churn_encoder = OneHotEncoder(CATEGORICAL).fit(train_customers)
model = LogisticRegression(max_iter=1000).fit(model_input(train_customers), y_train)
print(isinstance(churn_encoder, Transformer))

Now the real test of the warm-up bug. Score the two-year customers **alone** (a batch where `Contract` has only one value) and compare with their scores **inside the full batch**:

In [ ]:
two_year = new_customers[new_customers["Contract"] == "Two year"]

alone = model.predict_proba(model_input(two_year))[:, 1]
in_full_batch = pd.Series(
    model.predict_proba(model_input(new_customers))[:, 1], index=new_customers.index
)[two_year.index]

print(f"{len(two_year)} two-year customers")
print("identical scores alone and in the full batch:", np.allclose(alone, in_full_batch))
print(
    "columns the model sees, alone vs full batch identical:",
    model_input(two_year).columns.equals(model_input(new_customers).columns),
)

A customer's score no longer depends on who else is in the file. In notebook 10 we will turn exactly this comparison into an automated regression test.

### Check your understanding

```python
class LogTransform(Transformer):
    def transform(self, X):
        return np.log1p(X)

step = LogTransform()
```

What happens, and how would you fix it if the step really has nothing to learn?

<details><summary>Answer</summary>

`TypeError` at `LogTransform()`: `fit` is still abstract. A step without parameters still has to honour the contract, so it implements a `fit` that learns nothing and returns `self`. That is exactly what scikit-learn's `FunctionTransformer` does. The contract makes every step interchangeable, even the trivial ones.
</details>

## Exercises

### Exercise 4.1 · Core: a `MedianImputer`

Next month's batch has gaps: some customers have no `MonthlyCharges` because of a billing-system migration. You must fill them, and the fill value must come from the **training data**. If you used the median of the new batch, a batch of mostly expensive fibre customers would get a different fill value than a batch of mostly cheap DSL customers: the same bug as before, in a new costume.

Write `MedianImputer(Transformer)`:

- `__init__(self, columns)` stores the columns to impute,
- `fit` learns the median of each column and stores it internally,
- `transform` returns a **copy** of X with missing values in those columns replaced by the learned medians; it raises `RuntimeError` if called before `fit`,
- a read-only property `medians` returns the learned medians as a dict.

In [ ]:
class MedianImputer(Transformer):
    def __init__(self, columns: list[str]):
        raise NotImplementedError("Your code here")

    def fit(self, X: pd.DataFrame) -> "MedianImputer":
        raise NotImplementedError("Your code here")

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        raise NotImplementedError("Your code here")


batch = new_customers.head(6).copy()
batch.loc[batch.index[[1, 4]], "MonthlyCharges"] = np.nan

imputer = MedianImputer(["MonthlyCharges"]).fit(train_customers)
imputer.transform(batch)[["Contract", "MonthlyCharges"]]

In [ ]:
# Check cell
train_median = train_customers["MonthlyCharges"].median()
filled = imputer.transform(batch)
assert filled["MonthlyCharges"].isna().sum() == 0
assert filled["MonthlyCharges"].iloc[1] == train_median  # the training median ...
assert filled["MonthlyCharges"].iloc[4] == train_median
assert batch["MonthlyCharges"].isna().sum() == 2  # ... and the input is untouched
assert imputer.medians == {"MonthlyCharges": train_median}
try:
    MedianImputer(["MonthlyCharges"]).transform(batch)
    raise AssertionError("transform before fit should raise RuntimeError")
except RuntimeError:
    pass
print("All checks passed. Fill value:", train_median)

<details><summary>Hint</summary>

`X[self.columns].median()` gives a Series of medians; `.to_dict()` turns it into a dict. `X.fillna(value=a_dict)` fills each column with its own value and returns a new DataFrame. Follow the encoder: `_medians` and `_fitted` internally, `medians` as a property returning a copy.
</details>

<details><summary>Solution</summary>

```python
class MedianImputer(Transformer):
    def __init__(self, columns: list[str]):
        self.columns = list(columns)
        self._medians: dict[str, float] = {}
        self._fitted = False

    @property
    def medians(self) -> dict[str, float]:
        return dict(self._medians)

    def fit(self, X: pd.DataFrame) -> "MedianImputer":
        self._medians = X[self.columns].median().to_dict()
        self._fitted = True
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if not self._fitted:
            raise RuntimeError("MedianImputer must be fitted before transform()")
        return X.fillna(value=self._medians)
```

`fillna(value=dict)` only touches the listed columns and returns a new DataFrame, so the input stays unchanged.
</details>

### Exercise 4.2 · Stretch: what about categories nobody has seen?

The company launches a "Three year" contract. Our encoder silently encodes such customers as all zeros in every `Contract=...` column. Sometimes that is acceptable (the model treats them like "none of the known contracts"), sometimes you would rather stop. scikit-learn's `OneHotEncoder` has a `handle_unknown` parameter for this choice.

Write `SafeOneHotEncoder(OneHotEncoder)` with `__init__(self, columns, handle_unknown="ignore")`:

- `handle_unknown` must be `"ignore"` or `"error"`; anything else raises `ValueError` **in `__init__`** (fail early),
- in `transform`, if a column contains categories not seen during fit:
  - `"error"`: raise `ValueError` naming the column and the unseen values,
  - `"ignore"`: call `warnings.warn(...)` with the same information and continue (all zeros),
- reuse the parent's logic with `super()`; do not copy the encoding loop.

In [ ]:
import warnings


class SafeOneHotEncoder(OneHotEncoder):
    def __init__(self, columns: list[str], handle_unknown: str = "ignore"):
        raise NotImplementedError("Your code here")

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        raise NotImplementedError("Your code here")


launch = pd.DataFrame({"Contract": ["Three year", "One year"]})
safe = SafeOneHotEncoder(["Contract"]).fit(train)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    encoded = safe.transform(launch)
print([str(w.message) for w in caught])
encoded

In [ ]:
# Check cell
assert len(caught) == 1 and "Three year" in str(caught[0].message)
assert encoded.loc[0].sum() == 0 and encoded.loc[1, "Contract=One year"] == 1
strict = SafeOneHotEncoder(["Contract"], handle_unknown="error").fit(train)
try:
    strict.transform(launch)
    raise AssertionError("expected ValueError for an unseen category")
except ValueError as exc:
    assert "Three year" in str(exc)
try:
    SafeOneHotEncoder(["Contract"], handle_unknown="skip")
    raise AssertionError("expected ValueError for an invalid handle_unknown")
except ValueError:
    pass
print("All checks passed")

<details><summary>Hint</summary>

In `__init__`, validate first, then call `super().__init__(columns)` and store the option. In `transform`, `self._categories` tells you the known categories; `~X[col].isin(known)` selects unseen rows. After the checks, `return super().transform(X)`.
</details>

<details><summary>Solution</summary>

```python
class SafeOneHotEncoder(OneHotEncoder):
    def __init__(self, columns: list[str], handle_unknown: str = "ignore"):
        if handle_unknown not in ("ignore", "error"):
            raise ValueError(f"handle_unknown must be 'ignore' or 'error', got {handle_unknown!r}")
        super().__init__(columns)
        self.handle_unknown = handle_unknown

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if self.is_fitted:
            for col, known in self._categories.items():
                unseen = sorted(set(X[col]) - set(known))
                if not unseen:
                    continue
                message = f"{col}: categories not seen during fit: {unseen}"
                if self.handle_unknown == "error":
                    raise ValueError(message)
                warnings.warn(message + " (encoded as all zeros)", stacklevel=2)
        return super().transform(X)
```

The `if self.is_fitted` guard lets the parent raise its usual "must be fitted" error. A subclass may read `self._categories`: the underscore means "internal to this class family", and subclasses are part of the family. `warnings.warn` (rather than `print`) lets the user of your class decide whether to show, log, silence, or turn the warning into an error.
</details>

### Exercise 4.3 · Challenge: a chain of transformers that is itself a transformer

Write `TransformerChain(Transformer)`, constructed with a list of steps:

- `fit` fits each step on the output of the previous step (use `fit_transform`),
- `transform` applies the already fitted steps in order (raise `RuntimeError` if not fitted),
- because it is a `Transformer`, a chain can contain another chain.

This is the **Composite** pattern: a group of objects that can be used exactly like a single object. Then answer: how does your class compare with scikit-learn's `Pipeline`? Name at least two things `Pipeline` does that yours does not.

In [ ]:
class DropColumns(Transformer):
    """Remove columns the model must not see. Nothing to learn."""

    def __init__(self, columns: list[str]):
        self.columns = list(columns)

    def fit(self, X: pd.DataFrame) -> "DropColumns":
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return X.drop(columns=self.columns)

In [ ]:
class TransformerChain(Transformer):
    def __init__(self, steps: list[Transformer]):
        raise NotImplementedError("Your code here")

    def fit(self, X: pd.DataFrame) -> "TransformerChain":
        raise NotImplementedError("Your code here")

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        raise NotImplementedError("Your code here")


chain = TransformerChain([DropColumns(["gender"]), OneHotEncoder(["Contract", "InternetService"])])
chain.fit(train_customers).transform(two_year).head(3)

In [ ]:
# Check cell
expected = OneHotEncoder(["Contract", "InternetService"]).fit_transform(
    train_customers.drop(columns=["gender"])
)
pd.testing.assert_frame_equal(chain.fit_transform(train_customers), expected)
assert chain.transform(two_year).columns.equals(expected.columns)  # stable columns
assert isinstance(chain, Transformer)
nested = TransformerChain([chain, DropColumns(["PaymentMethod"])])
assert "PaymentMethod" not in nested.fit_transform(train_customers).columns
try:
    TransformerChain([DropColumns(["gender"])]).transform(train_customers)
    raise AssertionError("transform before fit should raise RuntimeError")
except RuntimeError:
    pass
print("All checks passed")

<details><summary>Hint</summary>

Keep `self._steps = list(steps)` and `self._fitted = False`. In `fit`, loop with a variable `current = X` and replace it by `step.fit_transform(current)` at each step. `transform` is the same loop with `step.transform`.
</details>

<details><summary>Solution</summary>

```python
class TransformerChain(Transformer):
    def __init__(self, steps: list[Transformer]):
        self._steps = list(steps)
        self._fitted = False

    def fit(self, X: pd.DataFrame) -> "TransformerChain":
        current = X
        for step in self._steps:
            current = step.fit_transform(current)
        self._fitted = True
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if not self._fitted:
            raise RuntimeError("TransformerChain must be fitted before transform()")
        current = X
        for step in self._steps:
            current = step.transform(current)
        return current
```

Some things scikit-learn's `Pipeline` adds: the last step may be a model, so the whole pipeline has `predict` and `score`; steps have names and can be looked up (`pipe["scaler"]`, `pipe[0]`, `len(pipe)`, see notebook 06); `get_params`/`set_params` with `step__param` names, so `GridSearchCV` can tune parameters of inner steps; cloning for cross-validation; caching of fitted steps (`memory=`). Combined with `ColumnTransformer` it also applies different steps to different columns. In real projects, use `Pipeline`; writing your own once is how you understand it.
</details>

## Summary

- A preprocessing step **learns in `fit`, replays in `transform`**. Re-fitting on each new batch makes a customer's prediction depend on who else is in the file.
- Learned parameters are **state**. Keep it internal (`_name`), expose what callers need through **properties**, and hand out copies of mutable internals.
- A leading underscore is a convention, not a lock. It defines what is interface and what may change.
- **Validating setters** keep invariants (a threshold is a probability) in exactly one place; let `__init__` use the setter too.
- An **ABC** writes the contract down: forgetting a required method fails when the object is created, and shared behaviour (`fit_transform`) is written once.

**Common pitfalls**

- Returning a reference to an internal list or dict from a method or property.
- Computing statistics (medians, means, categories) on the batch being transformed instead of on the training data.
- Believing `_private` or `__private` makes data secret. It does not; it communicates intent.

## Going further

Not part of the live session.

- **Template method.** In our ABC every subclass repeats the "fitted?" bookkeeping. `src/churn/features.py` moves it into the base class: the public `fit` and `transform` are concrete methods that handle the flag and the error, and they call abstract `_fit` and `_transform` that subclasses implement. Subclasses cannot forget the check any more. Compare the two designs.
- **Abstract properties**: `@property` stacked on `@abstractmethod` forces subclasses to provide an attribute-like value.
- **`__slots__`** fixes the set of attributes an object may have (fewer typos, less memory). Rarely needed in DS code.
- **scikit-learn's version**: `sklearn.preprocessing.OneHotEncoder(handle_unknown="ignore")` together with `ColumnTransformer` and `Pipeline` is what you should use in production. Notebook X2 shows how to make your own transformers scikit-learn compatible.

Resources:

- `abc` module: https://docs.python.org/3/library/abc.html
- `property` built-in: https://docs.python.org/3/library/functions.html#property
- scikit-learn `OneHotEncoder`: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html